In [51]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import  OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

In [52]:
df = pd.read_csv('train.csv')

In [53]:
df.head()

,id,road_type,num_lanes,curvature,speed_limit,lighting,weather,road_signs_present,public_road,time_of_day,holiday,school_season,num_reported_accidents,accident_risk
0,0,urban,2,0.06,35,daylight,rainy,False,True,afternoon,False,True,1,0.13
1,1,urban,4,0.99,35,daylight,clear,True,False,evening,True,True,0,0.35
2,2,rural,4,0.63,70,dim,clear,False,True,morning,True,False,2,0.30
3,3,highway,4,0.07,35,dim,rainy,True,True,morning,False,False,1,0.21
4,4,rural,1,0.58,60,daylight,foggy,False,False,evening,True,False,1,0.56


In [54]:
df = df.drop(
    columns=[
        "road_type",
        "num_lanes",
        "school_season",
        "time_of_day",
        "road_signs_present",
    ]
)

In [55]:
X = df.drop(columns=['id', 'accident_risk'])
y = df['accident_risk']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

categorical_features = ['lighting', 'weather']
numerical_features = ['curvature', 'speed_limit', 'num_reported_accidents']
bool_features = ['public_road', 'holiday']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numerical_features + bool_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

In [56]:
numerical_features = ['curvature', 'speed_limit', 'num_reported_accidents']
categorical_features = ['lighting', 'weather']
boolean_features = ['public_road', 'holiday']


numerical_transformer = SimpleImputer(strategy="mean")
categorical_transformer = SimpleImputer(strategy="most_frequent")
boolean_transformer = SimpleImputer(strategy="most_frequent")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features),
        ("bool", boolean_transformer, boolean_features),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('bool', 'passthrough', boolean_features)
    ],
    remainder='drop'
)

In [57]:
Enchanted_forest = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(
        n_estimators=100,   
        max_depth=10,       
        random_state=42     
    ))
])

Enchanted_forest.fit(X_train, y_train)

predictions = Enchanted_forest.predict(X_train)
rmse = np.sqrt(mean_squared_error(y_train, predictions))

print(f"Random Forest RMSE: {rmse:.4f}")

Random Forest RMSE: 0.0556


## **Enchanted forest RMSE 0.0556**

In [58]:
final_df = pd.read_csv('test.csv')

Enchanted_forest_predictions = Enchanted_forest.predict(final_df.drop(columns=['id']))

submission = pd.DataFrame({
    'id': final_df['id'],
    'accident_risk': Enchanted_forest_predictions
})

submission.to_csv('submission_enchanted_forest.csv', index=False)

## **Fine-tune the model**

In [59]:
# base_pipeline = Pipeline(steps=[
#     ('preprocessor', preprocessor),
#     ('regressor', RandomForestRegressor(random_state=42))
# ])

# param_grid = {
#     'regressor__n_estimators': [100, 200, 300],
#     'regressor__max_depth': [None, 10, 20],
#     'regressor__min_samples_split': [2, 5, 10],
#     'regressor__max_features': ['sqrt', 'log2']
# }

# grid_search = GridSearchCV(
#     estimator=base_pipeline,
#     param_grid=param_grid,
#     cv=5,
#     scoring='neg_root_mean_squared_error',
#     n_jobs=-1
# )

# grid_search.fit(X_train, y_train)

# Enchanted_forest_finetuned = grid_search.best_estimator_
# y_pred = Enchanted_forest_finetuned.predict(X_test)
# rmse = np.sqrt(mean_squared_error(y_test, y_pred))

# print(f"Best Parameters: {grid_search.best_params_}")
# print(f"Final RMSE: {rmse:.4f}")

In [66]:
import joblib
joblib.dump(Enchanted_forest, 'Enchanted_forest.joblib')

['Enchanted_forest.joblib']

In [67]:
test_data = [
    [12, 90, 4, 'night', 'foggy', 1, 0],  
    [np.nan, np.nan, 0, 'night', np.nan, 1, np.nan]  
]

feature_names = ['curvature', 'speed_limit', 'num_reported_accidents', 'lighting', 'weather', 'public_road', 'holiday']
df_test = pd.DataFrame(test_data, columns=feature_names)

display(df_test)

try:
    probabilities = Enchanted_forest.predict(df_test)
    
    print("pass")
    

        
except ValueError as e:
    print("fail")
    print(f"{e}")

,curvature,speed_limit,num_reported_accidents,lighting,weather,public_road,holiday
0,12.0,90.0,4,night,foggy,1,0.0
1,NaN,NaN,0,night,NaN,1,NaN


pass


In [62]:
test_data = [
    [12, 90, 4, 'night', 'foggy', 1, 0],  
    [np.nan, np.nan, 0, 'night', np.nan, 1, np.nan]  
]

feature_names = ['curvature', 'speed_limit', 'num_reported_accidents', 'lighting', 'weather', 'public_road', 'holiday']
df_test = pd.DataFrame(test_data, columns=feature_names)

# display(df_test)

try:
    probabilities = Enchanted_forest.predict(df_test)
    print("pass")


except ValueError as e:
    print("fail")
    print(f"Error details: {e}")



pass


In [63]:
probabilities

array([0.82194902, 0.34508603])

In [68]:
model_path = 'Enchanted_forest.joblib' 

try:
    loaded_model = joblib.load(model_path)
    
    test_data = [
        [12, 90, 4, 'night', 'foggy', 1, 0],  
        [np.nan, np.nan, 0, 'night', np.nan, 1, np.nan]  
    ]

    feature_names = ['curvature', 'speed_limit', 'num_reported_accidents', 'lighting', 'weather', 'public_road', 'holiday']
    df_test = pd.DataFrame(test_data, columns=feature_names)
    
    display(df_test)
    
    probabili = loaded_model.predict(df_test)    
        
except FileNotFoundError:
    print("error")
except Exception as e:
    print("error")
    print(f"{e}")


,curvature,speed_limit,num_reported_accidents,lighting,weather,public_road,holiday
0,12.0,90.0,4,night,foggy,1,0.0
1,NaN,NaN,0,night,NaN,1,NaN


In [69]:
probabili

array([0.82194902, 0.34508603])

In [73]:
data = pd.read_csv('captured_road_data.csv')
data.head()

,curvature,speed_limit,num_reported_accidents,lighting,weather,public_road,holiday
0,20.0,100.0,0.0,daylight,clear,1,0


In [74]:
p = loaded_model.predict(data)    
p

array([0.47062651])